<a href="https://colab.research.google.com/github/nateraw/download-musiccaps-dataset/blob/main/download_musiccaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading Clips from the MusicCaps Dataset

In this notebook, we see how you can use `yt-dlp` to download clips from the [MusicCaps](https://huggingface.co/datasets/google/MusicCaps) dataset from Google. The MusicCaps dataset contains music and their associated text captions. You could use a dataset like this to train a text-to-audio generation model 😉. 

Once we've downloaded the clips, we'll explore them using a [Gradio](https://gradio.app/) interface.

If you like this notebook:

  - consider giving the [repo](https://github.com/nateraw/download-musiccaps-dataset) a star ⭐️
  - consider following me on Github [@nateraw](https://github.com/nateraw)

In [1]:
%%capture
! pip install datasets[audio] yt-dlp

from google.colab import drive
drive.mount('/content/gdrive')

In [2]:
import subprocess
import os
from pathlib import Path
import time

from datasets import load_dataset, Audio


def download_clip(
    video_identifier,
    output_filename,
    start_time,
    end_time,
    tmp_dir='/tmp/musiccaps',
    num_attempts=2,
    url_base='https://www.youtube.com/watch?v='
):
    status = False

    command = f"""
        yt-dlp -i --sleep-interval 2 --max-sleep-interval 5 --cookies /content/gdrive/MyDrive/cookies.txt --quiet --force-keyframes-at-cuts --no-warnings -x --audio-format wav -f bestaudio -o "{output_filename}" --download-sections "*{start_time}-{end_time}" {url_base}{video_identifier}
    """.strip()
    # command = [
    #     "yt-dlp",
    #     "--cookies", "cookies.txt",
    #     # "--quiet",
    #     "--force-keyframes-at-cuts",
    #     # "--no-warnings",
    #     "-x",
    #     "--audio-format", "wav",
    #     "-f", "bestaudio",
    #     "-o", output_filename,
    #     "--download-sections", f"*{start_time}-{end_time}",
    #     f"{url_base}{video_identifier}"
    # ]

    attempts = 0
    while True:
        try:
            output = subprocess.check_output(command, shell=True,
                                                stderr=subprocess.STDOUT)
        except subprocess.CalledProcessError as err:
            attempts += 1
            if attempts == num_attempts:
                return status, err.output
        else:
            break

    # Check if the video was successfully saved.
    status = os.path.exists(output_filename)
    return status, 'Downloaded'


def main(
    data_dir: str,
    sampling_rate: int = 44100,
    limit: int = None,
    num_proc: int = 1,
    writer_batch_size: int = 1000,
):
    """
    Download the clips within the MusicCaps dataset from YouTube.
    Args:
        data_dir: Directory to save the clips to.
        sampling_rate: Sampling rate of the audio clips.
        limit: Limit the number of examples to download.
        num_proc: Number of processes to use for downloading.
        writer_batch_size: Batch size for writing the dataset. This is per process.
    """

    ds = load_dataset('google/MusicCaps', split='train')
    start_idx = 0
    if limit is not None:
        print(f"Limiting to {limit} examples")
        ds = ds.select(range(limit))
    else:
        print(f"Starting at sample {start_idx}, using remaining samples")
        ds = ds.select(range(start_idx, len(ds)))

    data_dir = Path(data_dir)
    data_dir.mkdir(exist_ok=True, parents=True)

    def process(example):
        outfile_path = str(data_dir / f"{example['ytid']}.wav")
        status = True
        if not os.path.exists(outfile_path):
            status = False
            status, log = download_clip(
                example['ytid'],
                outfile_path,
                example['start_s'],
                example['end_s'],
            )
            # time.sleep(1)
            if not status:
                # Print failure info immediately
                print(f"FAILED to download {example['ytid']}")
                print(log.decode("utf-8") if isinstance(log, bytes) else log)

        example['audio'] = outfile_path
        example['download_status'] = status
        return example

    ds = ds.map(
        process,
        num_proc=num_proc,
        writer_batch_size=writer_batch_size,
        keep_in_memory=False
    )
    
    ds = ds.filter(lambda x: x["download_status"])

    return ds.cast_column('audio', Audio(sampling_rate=sampling_rate))

## Load the Dataset

Here we are limiting to the first 32 examples. Since Colab is constrained to 2 cores, downloading the whole dataset here would take hours.

When running this on your own machine:
  - you can set `limit=None` to download + process the full dataset. Feel free to do that here in Colab, it'll just take a long time.
  - you should increase the `num_proc`, which will speed things up substantially
  - If you run out of memory, try reducing the `writer_batch_size`, as by default, it will keep 1000 examples in memory *per worker*.

In [3]:
ds = main('/content/gdrive/MyDrive/music_data', num_proc=2)
print("Final dataset size:", len(ds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

musiccaps-public.csv:   0%|          | 0.00/2.94M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5521 [00:00<?, ? examples/s]

Starting at sample 0, using remaining samples


Map (num_proc=2):   0%|          | 0/5521 [00:00<?, ? examples/s]

FAILED to download SLq-Co_szYo
ERROR: [youtube] SLq-Co_szYo: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

FAILED to download -0xzrMun0Rs
ERROR: [youtube] -0xzrMun0Rs: This video is not available

FAILED to download SMjYheNdAzg
ERROR: [youtube] SMjYheNdAzg: This video is not available

FAILED to download -FlvaZQOr2I
ERROR: [youtube] -FlvaZQOr2I: This video is not available

FAILED to download Sp3T_x3SGzQ
ERROR: [youtube] Sp3T_x3SGzQ: This video is not available

FAILED to download -IZbvEO9wzU
ERROR: [youtube] -IZbvEO9wzU: This video is not available

FAILED to download T6iv9GFIVyU
ERROR: [youtube] T6iv9GFIVyU: Video unavailable. This video is no longer av

Filter:   0%|          | 0/5521 [00:00<?, ? examples/s]

Final dataset size: 4204


In [ ]:
# save the dataset

ds.save_to_disk("/content/gdrive/MyDrive/musiccaps_with_audio")


Saving the dataset (0/16 shards):   0%|          | 0/4204 [00:00<?, ? examples/s]